## Creating MCP-Langchain Agent

In [1]:
import os 
from dotenv import load_dotenv 
load_dotenv() 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


import warnings

warnings.filterwarnings("ignore",category=DeprecationWarning)

from langchain.chat_models import init_chat_model

gemma = init_chat_model(model="gemma4:latest", model_provider="ollama")

#llm = init_chat_model(model="qwen/qwen3-32b", model_provider="Groq")
llm_primary = init_chat_model(model="llama-3.3-70b-versatile", model_provider="Groq")
llm_fallback_1 = init_chat_model(model="gpt-5.4-nano", model_provider="OpenAI")
llm_fallback_2 = init_chat_model(model="gpt-5.4-mini", model_provider="OpenAI")


In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

## Connect your client with the MongoDB-MCP-server 

In [3]:
client = MultiServerMCPClient(
    {
  "Course_assistant": {
      "transport": "stdio",
      "command": "uv",
      "args": [
        "run",
        "python",
        "server.py"
      ],
      "cwd": "/Users/nali/Documents/YTLLMs/AgenticAI/Agents/MCP-Server"
    
  }

})

In [4]:
tools = await client.get_tools()

In [5]:
for tool in tools: 
    print(tool)

name='get_course_list' description='return list of available courses' args_schema={'properties': {}, 'title': 'get_course_listArguments', 'type': 'object'} handle_tool_error=<function _handle_mcp_tool_error at 0x115b33a60> response_format='content_and_artifact' coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x114fabec0>
name='get_course_details' description='Return detailed course information of a given course' args_schema={'properties': {'course_id': {'title': 'Course Id', 'type': 'string'}}, 'required': ['course_id'], 'title': 'get_course_detailsArguments', 'type': 'object'} handle_tool_error=<function _handle_mcp_tool_error at 0x115b33a60> response_format='content_and_artifact' coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x1068e0720>


## Resources provide context to the user or LLMs 

In [30]:
#get all resources
resources = await client.get_resources(server_name="Course_assistant")
print(resources)

[Blob 4683500912]


In [38]:
## get resource for a specific course 
resources = await client.get_resources(server_name="Course_assistant", uris="course://cs466")
print(resources)

[Blob 4679894992]


In [39]:
for resource in resources:
    print(resource)

metadata={'uri': 'course://cs466'} data='{\n  "course_id": "CS 466",\n  "section": "01",\n  "title": "Natural Language Processing & Large Language Models (NLP & LLMs)",\n  "short_title": "NLP & LLMs",\n  "semester": "Fall 2026",\n  "credit_hours": 3.0,\n  "meeting": {\n    "days": [\n      "Monday",\n      "Wednesday"\n    ],\n    "time": "3:00 PM - 4:15 PM",\n    "location": "BR 160"\n  },\n  "instructor": {\n    "name": "Dr. G. G. Md. Nawaz Ali",\n    "office": "Bradley Hall 297",\n    "office_hours": "Tuesday, Wednesday, and Thursday, 1:00 PM - 2:00 PM"\n  },\n  "course_web": "https://learn.bradley.edu/",\n  "prerequisites": "CS 461 or CS 462 or consent of instructor",\n  "description": "An in-depth, hands-on study of natural language processing and large language models, progressing from foundational NLP to transformer-based models and production-style AI systems. The course emphasizes practical implementation, evaluation, Retrieval-Augmented Generation, tool-using LLMs, multi-agen

In [40]:
context = "\n\n".join(
    f"{resource.as_string()}"
    for resource in resources
)

In [19]:
context

'{\n  "course_id": "CS 466",\n  "section": "01",\n  "title": "Natural Language Processing & Large Language Models (NLP & LLMs)",\n  "short_title": "NLP & LLMs",\n  "semester": "Fall 2026",\n  "credit_hours": 3.0,\n  "meeting": {\n    "days": [\n      "Monday",\n      "Wednesday"\n    ],\n    "time": "3:00 PM - 4:15 PM",\n    "location": "BR 160"\n  },\n  "instructor": {\n    "name": "Dr. G. G. Md. Nawaz Ali",\n    "office": "Bradley Hall 297",\n    "office_hours": "Tuesday, Wednesday, and Thursday, 1:00 PM - 2:00 PM"\n  },\n  "course_web": "https://learn.bradley.edu/",\n  "prerequisites": "CS 461 or CS 462 or consent of instructor",\n  "description": "An in-depth, hands-on study of natural language processing and large language models, progressing from foundational NLP to transformer-based models and production-style AI systems. The course emphasizes practical implementation, evaluation, Retrieval-Augmented Generation, tool-using LLMs, multi-agent systems, and neuro-symbolic reasoning.

## Prompts help you to provide reuseable prompts for users to use 

In [33]:
prompts = await client.get_prompt(
    server_name="Course_assistant",
    prompt_name="all_course_summary")

In [41]:
##get promopt for a specific course
prompts = await client.get_prompt(
    server_name="Course_assistant",
    prompt_name="course_summary",
    arguments={
        "course_id":"CS 466"
    })

In [42]:
for prompt in prompts:
    print(prompt.content)

Using the course assistant tools and resources summarize CS 466 courses
    
    Include: 
    - Course description 
    - Course prerequisites
    - Course Learning Objectives 
    - Expected Student Outcomes 
    


## Create Langchain agent with mcp_tools

In [43]:
from langchain.agents import create_agent

prompt="""You are a helpful assistant. 
use tools for answering based on user query.
"""

agent= create_agent(
    model=  gemma,
    tools=tools,
    system_prompt=prompt
)

## Test the agent

#### Invoke using given prompt and resources 

In [ ]:
from langchain.messages import HumanMessage

In [ ]:
try: 
    result = await agent.ainvoke({
        "messages":[
            *prompts,
            HumanMessage(content=f"Course context: {context}")
    ]
    })
except Exception as e:
    print(f"Invoke error. {e}")

In [45]:
print(result["messages"][-1].content)

Based on the provided course context, here is a summary of CS 466: Natural Language Processing & Large Language Models (NLP & LLMs).

***

### **CS 466: Natural Language Processing & Large Language Models (NLP & LLMs)**

#### **📚 Course Description**
This is an in-depth, hands-on study of natural language processing and large language models. The course progresses from foundational NLP concepts to modern, transformer-based models and production-style AI systems. The curriculum places a strong emphasis on practical implementation, covering topics such as evaluation, Retrieval-Augmented Generation (RAG), tool-using LLMs, multi-agent systems, and neuro-symbolic reasoning.

#### **📝 Course Prerequisites**
*   CS 461 **OR** CS 462 **OR** consent of the instructor.

#### **🎯 Course Learning Objectives**
Upon completing this course, students will be able to:
*   Explain the linguistic, statistical, and neural foundations of NLP.
*   Implement core NLP pipelines using Python and modern librari

### More test queries

In [46]:
user_query = "How many courses are there? Give there course code and title."

In [47]:
from langchain.messages import SystemMessage,HumanMessage

try:
    result = await agent.ainvoke({
        "messages":[
            SystemMessage(content="You a helpful assistant."),
            HumanMessage(content=user_query)
        ]
    })
except Exception as e:
    print(f"Error happened during invoke. {e}")

In [48]:
print(result["messages"][-1].content)

There are 2 courses available:

1. **CS 111**: AI for All - Artificial Intelligence for Life, Society, and Disciplines
2. **CS 466**: Natural Language Processing & Large Language Models (NLP & LLMs)


In [9]:
user_query= """What are the software requirements for CS 466"""

In [10]:
user_query = """What are the 3 course outcomes for CS 111. What is the grading criteria?"""